# Model prototyping and offline evaluation

Train/test split, then Precision@K, Recall@K, MAP@K for baseline, CF, content, and hybrid.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from recommender.data_loading import load_movielens
from recommender.preprocessing import prepare_interaction_data
from recommender.features import build_genre_features
from recommender.models import PopularityRecommender, ItemItemRecommender, ContentRecommender, HybridRecommender
from recommender.evaluation import precision_at_k, recall_at_k, average_precision_at_k, map_at_k
from recommender import config

In [ ]:
ratings, movies = load_movielens()
interaction = prepare_interaction_data(ratings, movies)

# Simple train/test: for each user, leave out last 5 ratings as holdout
sort_cols = [config.USER_ID_COL, "timestamp"] if "timestamp" in interaction.ratings.columns else [config.USER_ID_COL]
ratings_sorted = interaction.ratings.sort_values(sort_cols)
holdout = ratings_sorted.groupby(config.USER_ID_COL).tail(5)
# Train = all ratings not in holdout (merge on user+movie, keep left_only)
merged = ratings_sorted.merge(
    holdout[[config.USER_ID_COL, config.ITEM_ID_COL]],
    on=[config.USER_ID_COL, config.ITEM_ID_COL],
    how="left",
    indicator=True,
)
train = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])

per_user_holdout = holdout.groupby(config.USER_ID_COL)[config.ITEM_ID_COL].apply(set).to_dict()
print("Train size:", len(train), "Holdout size:", len(holdout))

In [ ]:
# Build models on full interaction (for simplicity); in production you'd train on train only
content_feat = build_genre_features(interaction.movies)
baseline = PopularityRecommender(interaction.ratings, interaction.movies)
item_item = ItemItemRecommender(interaction); item_item.fit()
content_rec = ContentRecommender(content_feat)
hybrid = HybridRecommender(baseline, item_item, content_rec)

K = 20
id_to_idx = interaction.item_id_to_index
idx_to_id = interaction.index_to_item_id

def eval_one_user(uid, recommender_fn, holdout_set):
    recs = recommender_fn(uid)
    rel = holdout_set.get(uid, set())
    if not rel: return None
    return {
        "p": precision_at_k(recs, rel, K),
        "r": recall_at_k(recs, rel, K),
        "ap": average_precision_at_k(recs, rel, K),
    }

# Sample users for quick eval
sample_users = list(per_user_holdout.keys())[:200]
print("Sampled", len(sample_users), "users for evaluation.")

In [ ]:
# Baseline: recommend popularity (no user signal) -> same recs for everyone; still compute metrics
baseline_recs = baseline.recommend(genres=None, top_k=K)[config.ITEM_ID_COL].tolist()
results_baseline = [eval_one_user(uid, lambda _: baseline_recs, per_user_holdout) for uid in sample_users]
results_baseline = [r for r in results_baseline if r is not None]
if results_baseline:
    print("Baseline (popularity) P@20:", sum(r["p"] for r in results_baseline)/len(results_baseline))
    print("Baseline R@20:", sum(r["r"] for r in results_baseline)/len(results_baseline))
    print("Baseline AP@20:", sum(r["ap"] for r in results_baseline)/len(results_baseline))